# Knowledge distillation for ViT-based bicycle detection

Mehrdad Mirpourian

This notebook holds all three experiments from my final project on making a
Vision Transformer small enough to run on a car's blind-spot camera while
still catching cyclists. CIFAR-100 is relabelled into a binary task, bike
against everything else, which leaves about 1% positives. That imbalance is
the point: in traffic, the object you must not miss is the rare one.

The question is which distillation signal actually helps a ViT-Tiny student
under those conditions.

| Part | Experiment | Teacher | Student | What it asks |
| --- | --- | --- | --- | --- |
| 1 | I | ViT-Large | ViT-Tiny | Which KD signal works best across a large capacity gap: logits, features, or attention? |
| 2 | II | ViT-Base | ViT-Tiny | Does a smaller teacher change the answer? |
| 3 | III | ViT-Large via ViT-Small | ViT-Tiny | Does a teacher assistant in the middle help? |

Every run reports precision, recall and F1 for the bike class. Accuracy is
not used for anything: a model that always answers "no bike" scores 99%.

**Running this.** It needs a GPU; I ran it on Colab with an A100. Parts 1 and
3 together take around four hours, Part 2 about ninety minutes, mostly in the
ViT-Large forward passes. Cells are meant to run top to bottom. Every cell
below still carries the output from my own run, so the numbers can be read
without running anything.

The write-up in `report/` explains the method and discusses the results. A
summary table is at the bottom of this notebook.

In [1]:
# =========================================================
# Environment setup
# =========================================================

import sys

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from tqdm import tqdm

import timm  # pretrained ViT backbones

# fvcore is only used for the FLOP counts. Colab does not ship it.
if "fvcore" not in sys.modules:
    !pip install -q fvcore

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Fixed seeds. This makes a single run repeatable, but note that the
# dataloader workers and cuDNN kernels still introduce small variations
# between sessions.
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Everything that I tuned by hand lives in one place, so the three
# experiments below all run under the same budget.
config = {
    "image_size": 224,
    "batch_size": 32,
    "num_workers": 4,

    # Teacher fine-tuning
    "lr_teacher": 2e-5,
    "epochs_teacher": 4,

    # Student / KD runs
    "lr_student": 2e-4,
    "epochs_student": 18,

    "num_classes": 2,
    "temperature": 1.5,
}

print("Configuration:", config)

Using device: cuda
Configuration: {'image_size': 224, 'batch_size': 32, 'num_workers': 4, 'lr_teacher': 2e-05, 'epochs_teacher': 4, 'lr_student': 0.0002, 'epochs_student': 18, 'num_classes': 2, 'temperature': 1.5}


## Data

CIFAR-100 gives 60,000 images at 32x32, 50,000 for training and 10,000 for
test. I keep the bicycle class as positive and collapse the other 99 classes
into one negative class.

Three things in this cell are worth knowing about:

- The images are upsampled to 224x224 because that is what the pretrained ViT
  checkpoints expect. A 32x32 photo blown up to 224x224 is blurry, and that
  is part of what makes this a hard setting rather than a clean one.
- Training augmentation adds flips, a small rotation, colour jitter, light
  additive noise and random erasing, standing in for the noise and lighting
  variation of a real camera.
- Positives are oversampled 8x for the student runs, and the teachers train on
  the natural distribution. So the teacher's soft labels are not shaped by my
  resampling, which keeps the distillation signal honest.

In [2]:
# =========================================================
# CIFAR-100 -> binary "bike / no bike" data pipeline
# =========================================================

cifar100_mean = (0.5071, 0.4867, 0.4408)
cifar100_std = (0.2675, 0.2565, 0.2761)

# 1) Transforms.
#    The ViT checkpoints expect 224x224 inputs, so the 32x32 CIFAR images
#    are upsampled first. Validation and test only get resize + normalize.
val_test_transform = T.Compose([
    T.Resize((config["image_size"], config["image_size"])),
    T.ToTensor(),
    T.Normalize(cifar100_mean, cifar100_std),
])

# The training transform adds the augmentation described in the report:
# small geometric jitter, lighting changes, and a little pixel noise, which
# stand in for the camera noise of a real blind-spot camera.
train_transform = T.Compose([
    T.Resize((config["image_size"], config["image_size"])),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomCrop(config["image_size"], padding=4),
    T.RandomRotation(5),
    T.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.10,
    ),
    T.RandomGrayscale(p=0.05),
    T.ToTensor(),
    T.Normalize(cifar100_mean, cifar100_std),
    # Mild additive noise. Kept small on purpose: with a stronger setting the
    # teacher and the student see too different a view of the same image.
    T.Lambda(lambda x: x + 0.005 * torch.randn_like(x)),
    T.RandomErasing(p=0.15),
])

# 2) Datasets.
full_train = torchvision.datasets.CIFAR100(
    root="./data",
    train=True,
    download=True,
    transform=None
)

test_dataset = torchvision.datasets.CIFAR100(
    root="./data",
    train=False,
    download=True,
    transform=val_test_transform
)

# 3) Relabel: class 8 is the bicycle class, everything else is negative.
#    This leaves roughly 1% positives, which is the imbalance I want.
POS_CLASS = {8}

full_train.targets = [1 if y in POS_CLASS else 0 for y in full_train.targets]
test_dataset.targets = [1 if y in POS_CLASS else 0 for y in test_dataset.targets]

# 4) Train / validation split out of the 50k training images.
num_train = len(full_train)
val_size = 5000
train_size = num_train - val_size

rng = torch.Generator().manual_seed(41)
train_subset, val_subset = torch.utils.data.random_split(
    full_train, [train_size, val_size], generator=rng
)

train_indices = list(train_subset.indices)
val_indices = list(val_subset.indices)

# NOTE: both subsets wrap the same CIFAR100 object, so these two lines set the
# transform of that one shared object rather than of each subset. The last
# assignment in this cell therefore wins for the validation split as well.
# See "Known issues" in the README; the test set is a separate object and is
# not affected.
train_subset.dataset.transform = train_transform
val_subset.dataset.transform = val_test_transform

# 5) Oversample the positives for the student runs.
#    8x is a compromise: enough signal for the rare class, but still far from
#    a 50/50 split, so the loaders stay close to the real distribution.
train_labels = [full_train.targets[i] for i in train_indices]

pos_indices = [i for i in train_indices if full_train.targets[i] == 1]
neg_indices = [i for i in train_indices if full_train.targets[i] == 0]

balanced_indices = neg_indices + pos_indices * 8
np.random.shuffle(balanced_indices)

balanced_train_subset = Subset(full_train, balanced_indices)
balanced_train_subset.dataset.transform = train_transform

# 6) Loaders.
train_loader = DataLoader(
    balanced_train_subset,
    batch_size=config["batch_size"],
    shuffle=True,
    num_workers=config["num_workers"]
)

val_loader = DataLoader(
    val_subset,
    batch_size=config["batch_size"],
    shuffle=False,
    num_workers=config["num_workers"]
)

test_loader = DataLoader(
    test_dataset,
    batch_size=config["batch_size"],
    shuffle=False,
    num_workers=config["num_workers"]
)

print("Data pipeline ready (KD-safe oversampling).")
print(f"Train (balanced): {len(balanced_train_subset)}, Val: {len(val_subset)}, Test: {len(test_dataset)}")

Data pipeline ready (KD-safe oversampling).
Train (balanced): 48101, Val: 5000, Test: 10000


## Shared helpers

Everything below reuses this handful of functions, so the three experiments
really are run under the same conditions rather than under three slightly
different copies of the same loop. `evaluate` returns per-class metrics with
index 1 for the bike class, which is the number quoted everywhere.

In [ ]:
# =========================================================
# Small helpers used by every experiment below
# =========================================================


def model_cost(model):
    """Parameter count and inference FLOPs for one 224x224 image.

    fvcore prints a warning for the fused attention and GELU ops it does not
    recognise, so the GFLOP number is a lower bound. It is still useful for
    comparing the model sizes against each other.
    """
    param_count = sum(p.numel() for p in model.parameters())

    try:
        from fvcore.nn import FlopCountAnalysis
        dummy_input = torch.randn(1, 3, config["image_size"], config["image_size"]).to(device)
        flops = FlopCountAnalysis(model, dummy_input)
        total_flops = flops.total()
        flops_str = f"{total_flops/1e9:.2f} GFLOPs"
    except Exception as e:
        flops_str = f"FLOP analysis failed: {e}"

    return param_count, flops_str


def evaluate(model, loader):
    """Run the model over a loader and return accuracy and per-class metrics.

    Returns (accuracy, precision, recall, f1), where the last three are arrays
    indexed by class: [0] is non-bike and [1] is bike. Index 1 is the number
    that matters in this project.
    """
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, predicted = outputs.max(1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average=None, labels=[0, 1]
    )
    acc = accuracy_score(all_labels, all_preds)

    return acc, precision, recall, f1


def load_frozen_model(model_name, checkpoint_path):
    """Rebuild a fine-tuned model from a checkpoint and freeze it.

    Used for the teachers and for the teacher assistant. pretrained=False
    because the checkpoint holds every weight we want.
    """
    model = timm.create_model(
        model_name,
        pretrained=False,
        num_classes=config["num_classes"]
    )
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model = model.to(device)
    model.eval()

    for p in model.parameters():
        p.requires_grad = False

    return model


def new_student():
    """A fresh ViT-Tiny, pretrained on ImageNet, with a 2-class head."""
    return timm.create_model(
        "vit_tiny_patch16_224",
        pretrained=True,
        num_classes=config["num_classes"]
    ).to(device)


def kd_kl_loss(student_logits, teacher_logits, temperature):
    """Temperature-scaled KL between the teacher and student distributions.

    The T*T factor undoes the 1/T**2 that the softened softmax introduces in
    the gradients, so the KD term keeps the same scale as the CE term.
    """
    teacher_soft = F.softmax(teacher_logits / temperature, dim=1)
    student_log_soft = F.log_softmax(student_logits / temperature, dim=1)

    kl = F.kl_div(student_log_soft, teacher_soft, reduction="batchmean")

    return kl * (temperature * temperature)


def make_optimizer_and_schedule(params, lr, epochs):
    """AdamW plus a per-step cosine schedule.

    The schedule is stepped once per batch rather than once per epoch, which
    is what the ViT recipes assume.
    """
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=0.05)

    total_steps = len(train_loader) * epochs
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=total_steps,
        eta_min=1e-6
    )

    return optimizer, scheduler


# Loss weight on the rare class for the student runs. Milder than the
# teacher's weight because the student loader is already oversampled.
STUDENT_CLASS_WEIGHTS = torch.tensor([1.0, 4.0]).to(device)

---

# Part 1 - Experiment I: distilling from ViT-Large

The large-gap case: a 303M-parameter teacher and a 5.5M-parameter student,
roughly 55x apart. The student sees three different kinds of supervision in
turn, and in each case the only thing that changes is the distillation signal.

Order of this part: fine-tune the teacher, train the undistilled baseline,
then logit KD, feature KD and attention KD, each followed by its test-set
evaluation.

## 1.1 Teacher: ViT-Large

Four epochs are enough here. The teacher is fine-tuned on the natural 1%
distribution with a strong positive class weight, and the epoch with the best
bike-class F1 on validation is the one kept on disk. Every distillation run
afterwards reads that checkpoint, frozen.

In [4]:
# =========================================================
# Experiment I, step 1a: load the ViT-Large teacher
# =========================================================

# 1) Pretrained backbone, then swap the 1000-class head for a 2-class one.
teacher_model_large = timm.create_model(
    'vit_large_patch16_224',
    pretrained=True
)

teacher_model_large.head = nn.Linear(
    teacher_model_large.head.in_features,
    config["num_classes"]
)

teacher_model_large = teacher_model_large.to(device)

# 2) Freeze it for now. The next cell unfreezes it for fine-tuning; this is
#    just so an accidental forward pass here cannot change any weights.
teacher_model_large.eval()

for p in teacher_model_large.parameters():
    p.requires_grad = False

# 3) Cost of the teacher at inference time, for the tables in the report.
param_count, flops_str = model_cost(teacher_model_large)

print("ViT-Large teacher model initialized.")
print(f"Parameters: {param_count/1e6:.2f}M")
print(f"FLOPs: {flops_str}")
print("Classification head: Binary")
print(f"Device: {device}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
blocks.0.attn.attn_drop, blocks.1.attn.attn_drop, blocks.10.attn.attn_drop, blocks.11.attn.attn_drop, blocks.12.attn.attn_drop, blocks.13.attn.attn_drop, blocks.14.attn.attn_drop, blocks.15.attn.attn_drop, blocks.16.attn.attn_drop, blocks.17.attn.attn_drop, blocks.18.attn.attn_drop, blocks.19.attn.attn_drop, blocks.2.attn.attn_drop, blocks.20.attn.attn_drop, blocks.21.attn.attn_drop, blocks.22.attn.attn_drop, blocks.23.attn.attn_drop, blocks.3.attn.attn_drop, blocks.4.attn.attn

ViT-Large teacher model initialized.
Parameters: 303.30M
FLOPs: 59.70 GFLOPs
Classification head: Binary
Device: cuda


In [ ]:
# =========================================================
# Experiment I, step 1b: fine-tune the ViT-Large teacher
# =========================================================


def finetune_teacher(model, label, save_path, epochs):
    """Fine-tune a teacher on the hard labels and keep the best epoch.

    The same function fine-tunes the ViT-Base teacher in Experiment II, so the
    two teachers are trained under identical conditions and the comparison in
    the report is about capacity alone.
    """
    for p in model.parameters():
        p.requires_grad = True

    # The teacher trains on the natural 1% distribution. Only the students get
    # the oversampled loader, so the teacher's soft labels are not shaped by
    # my resampling choice.
    teacher_train_loader = DataLoader(
        train_subset,
        batch_size=config["batch_size"],
        shuffle=True,
        num_workers=config["num_workers"]
    )

    # A heavy positive weight is needed here because, without oversampling,
    # predicting "no bike" everywhere already gets 99% accuracy.
    class_weights = torch.tensor([1.0, 20.0]).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    optimizer = optim.AdamW(
        model.parameters(),
        lr=config["lr_teacher"],
        weight_decay=0.05
    )

    # Written out instead of using make_optimizer_and_schedule, because the
    # teacher runs on its own loader and so has a different number of steps.
    total_steps = len(teacher_train_loader) * epochs
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=total_steps,
        eta_min=1e-6
    )

    best_bike_f1 = 0.0
    global_step = 0

    for epoch in range(epochs):
        model.train()

        train_loss = 0.0
        correct = 0
        total = 0

        print(f"\n===== Epoch {epoch+1}/{epochs} =====")

        for images, labels in tqdm(teacher_train_loader):
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad(set_to_none=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            optimizer.step()
            scheduler.step()
            global_step += 1

            train_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

        avg_train_loss = train_loss / total
        train_acc = correct / total

        print(f"Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.4f}")

        # Validation after every epoch. Model selection is on the bike-class
        # F1, never on accuracy, which is meaningless at this imbalance.
        _, precision, recall, f1 = evaluate(model, val_loader)

        print("\n=== Validation Metrics ===")
        print(f"Class 1 (bike)     — Precision: {precision[1]:.4f}, Recall: {recall[1]:.4f}, F1: {f1[1]:.4f}")
        print(f"Class 0 (non-bike) — Precision: {precision[0]:.4f}, Recall: {recall[0]:.4f}, F1: {f1[0]:.4f}")

        if f1[1] > best_bike_f1:
            best_bike_f1 = f1[1]
            torch.save(model.state_dict(), save_path)
            print(f" Saved NEW BEST {label} teacher model")

    print("\n===== Teacher Fine-Tuning Complete =====")
    print(f"Best Bike F1 achieved: {best_bike_f1:.4f}")
    print(f"Best model saved at: {save_path}")

    return best_bike_f1


teacher_large_save_path = "vit_large_teacher_best.pth"
finetune_teacher(
    teacher_model_large,
    label="ViT-Large",
    save_path=teacher_large_save_path,
    epochs=config["epochs_teacher"],
)

In [6]:
# =========================================================
# Experiment I, step 1c: test the ViT-Large teacher
# =========================================================

best_model_path = "vit_large_teacher_best.pth"

try:
    state_dict = torch.load(best_model_path, map_location=device)
    teacher_model_large.load_state_dict(state_dict)
    print(f"\nLoaded BEST ViT-Large teacher model from: {best_model_path}")
except Exception as e:
    print(f"ERROR loading checkpoint: {e}")
    raise

# Frozen from here on: every distillation run below reads this teacher.
for p in teacher_model_large.parameters():
    p.requires_grad = False

teacher_model_large.eval()

acc, precision, recall, f1 = evaluate(teacher_model_large, test_loader)

print("\n===== FINAL TEST METRICS (ViT-Large TEACHER) =====")
print(f"Overall Accuracy            : {acc:.4f}")
print(f"Class 1 (bike)     — Precision: {precision[1]:.4f}, Recall: {recall[1]:.4f}, F1: {f1[1]:.4f}")
print(f"Class 0 (non-bike) — Precision: {precision[0]:.4f}, Recall: {recall[0]:.4f}, F1: {f1[0]:.4f}")
print(f"--> Final Bike-Class F1 (Teacher): {f1[1]:.4f}")


Loaded BEST ViT-Large teacher model from: vit_large_teacher_best.pth

===== FINAL TEST METRICS (ViT-Large TEACHER) =====
Overall Accuracy            : 0.9996
Class 1 (bike)     — Precision: 1.0000, Recall: 0.9600, F1: 0.9796
Class 0 (non-bike) — Precision: 0.9996, Recall: 1.0000, F1: 0.9998
--> Final Bike-Class F1 (Teacher): 0.9796


## 1.2 Baseline student: ViT-Tiny, no teacher

The control. Same architecture, same loader, same 18 epochs as every KD run
below, but trained on hard labels only. Its bike-class F1 is the bar the
distilled students have to clear.

In [7]:
# =========================================================
# Experiment I, step 2a: ViT-Tiny student baseline (no distillation)
# =========================================================

# This is the number every KD run has to beat: the same student architecture,
# the same loader, the same budget, trained only on the hard labels.


def train_baseline_student(save_path, epochs):
    """Train ViT-Tiny on the hard labels only, with no teacher involved."""
    student_model = new_student()

    param_count, flops_str = model_cost(student_model)

    print(f"Loaded ViT-Tiny student with {param_count/1e6:.2f}M params.")
    print(f"Student inference FLOPs: {flops_str}")

    criterion = nn.CrossEntropyLoss(weight=STUDENT_CLASS_WEIGHTS)

    optimizer, scheduler = make_optimizer_and_schedule(
        student_model.parameters(), config["lr_student"], epochs
    )

    best_bike_f1 = 0.0

    for epoch in range(epochs):
        student_model.train()
        train_loss = 0.0
        correct = 0
        total = 0

        print(f"\n===== Student Baseline (ViT-Tiny) — Epoch {epoch+1}/{epochs} =====")

        for images, labels in tqdm(train_loader):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad(set_to_none=True)

            outputs = student_model(images)
            loss = criterion(outputs, labels)

            loss.backward()

            torch.nn.utils.clip_grad_norm_(student_model.parameters(), 1.0)

            optimizer.step()
            scheduler.step()

            train_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

        avg_train_loss = train_loss / total
        train_acc = correct / total

        print(f"Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.4f}")

        val_acc, precision, recall, f1 = evaluate(student_model, val_loader)

        print("=== Validation Metrics ===")
        print(f"Accuracy                 : {val_acc:.4f}")
        print(f"Class 1 (bike)     — Precision: {precision[1]:.4f}, Recall: {recall[1]:.4f}, F1: {f1[1]:.4f}")
        print(f"Class 0 (non-bike) — Precision: {precision[0]:.4f}, Recall: {recall[0]:.4f}, F1: {f1[0]:.4f}")

        if f1[1] > best_bike_f1:
            best_bike_f1 = f1[1]
            torch.save(student_model.state_dict(), save_path)
            print(" Saved NEW BEST ViT-Tiny baseline model")

    print("\n===== Student Baseline Training Complete =====")
    print(f"Best Bike F1 (Student Baseline): {best_bike_f1:.4f}")
    print(f"Best model saved at: {save_path}")

    return best_bike_f1


student_save_path = "student_vit_tiny_baseline.pth"
train_baseline_student(student_save_path, config["epochs_student"])

blocks.0.attn.attn_drop, blocks.1.attn.attn_drop, blocks.10.attn.attn_drop, blocks.11.attn.attn_drop, blocks.2.attn.attn_drop, blocks.3.attn.attn_drop, blocks.4.attn.attn_drop, blocks.5.attn.attn_drop, blocks.6.attn.attn_drop, blocks.7.attn.attn_drop, blocks.8.attn.attn_drop, blocks.9.attn.attn_drop


Loaded ViT-Tiny student with 5.52M params.
Student inference FLOPs: 1.08 GFLOPs

===== Student Baseline (ViT-Tiny) — Epoch 1/18 =====


100%|██████████| 1504/1504 [01:06<00:00, 22.49it/s]

Train Loss: 0.1785 | Train Acc: 0.9675


=== Validation Metrics ===
Accuracy                 : 0.9918
Class 1 (bike)     — Precision: 0.9000, Recall: 0.3158, F1: 0.4675
Class 0 (non-bike) — Precision: 0.9922, Recall: 0.9996, F1: 0.9959
 Saved NEW BEST ViT-Tiny baseline model

===== Student Baseline (ViT-Tiny) — Epoch 2/18 =====


100%|██████████| 1504/1504 [01:06<00:00, 22.76it/s]

... (45 outputs left out)

=== Validation Metrics ===
Accuracy                 : 0.9960
Class 1 (bike)     — Precision: 0.8936, Recall: 0.7368, F1: 0.8077
Class 0 (non-bike) — Precision: 0.9970, Recall: 0.9990, F1: 0.9980

===== Student Baseline (ViT-Tiny) — Epoch 18/18 =====


100%|██████████| 1504/1504 [01:05<00:00, 22.83it/s]

Train Loss: 0.0019 | Train Acc: 0.9996


=== Validation Metrics ===
Accuracy                 : 0.9964
Class 1 (bike)     — Precision: 0.9535, Recall: 0.7193, F1: 0.8200
Class 0 (non-bike) — Precision: 0.9968, Recall: 0.9996, F1: 0.9982

===== Student Baseline Training Complete =====
Best Bike F1 (Student Baseline): 0.8257
Best model saved at: student_vit_tiny_baseline.pth


In [8]:
# =========================================================
# Experiment I, step 2b: test the ViT-Tiny baseline
# =========================================================

# Rebuilt from the checkpoint rather than reusing the object in memory, so
# that the numbers below come from the saved best epoch and not from the
# final one.
checkpoint_path = "student_vit_tiny_baseline.pth"
student_test_model = load_frozen_model("vit_tiny_patch16_224", checkpoint_path)

print(f"Loaded best ViT-Tiny student model from: {checkpoint_path}")

acc, precision, recall, f1 = evaluate(student_test_model, test_loader)

print("\n===== FINAL TEST METRICS (STUDENT: ViT-Tiny) =====")
print(f"Overall Accuracy        : {acc:.4f}")
print(f"Class 1 (bike)     — Precision: {precision[1]:.4f}, Recall: {recall[1]:.4f}, F1: {f1[1]:.4f}")
print(f"Class 0 (non-bike) — Precision: {precision[0]:.4f}, Recall: {recall[0]:.4f}, F1: {f1[0]:.4f}")
print(f"--> Final Bike-Class F1 (Student): {f1[1]:.4f}")

Loaded best ViT-Tiny student model from: student_vit_tiny_baseline.pth

===== FINAL TEST METRICS (STUDENT: ViT-Tiny) =====
Overall Accuracy        : 0.9973
Class 1 (bike)     — Precision: 0.9195, Recall: 0.8000, F1: 0.8556
Class 0 (non-bike) — Precision: 0.9980, Recall: 0.9993, F1: 0.9986
--> Final Bike-Class F1 (Student): 0.8556


## 1.3 Logit KD

The classic formulation: match the teacher's temperature-softened output
distribution with a KL term, keep the weighted cross-entropy on the true
labels, and split the two with alpha.

With only two classes there is not much "dark knowledge" in the teacher's
distribution, which made me expect this to be the weakest of the three. It
was not.

In [9]:
# =========================================================
# Experiment I, step 3: logit-based KD (ViT-Large -> ViT-Tiny)
# =========================================================

teacher_model = load_frozen_model("vit_large_patch16_224", "vit_large_teacher_best.pth")
print("Frozen ViT-Large teacher loaded.")


def train_logit_kd(teacher, save_path, epochs, temperature=4.0, alpha=0.7):
    """Train a ViT-Tiny student on soft teacher logits plus hard labels.

    alpha splits the two terms: alpha on the KD loss, 1 - alpha on the
    weighted cross-entropy. Returns the best bike-class F1 on validation.
    """
    kd_student = new_student()
    print("KD student = ViT-Tiny loaded.")

    ce_loss_fn = nn.CrossEntropyLoss(weight=STUDENT_CLASS_WEIGHTS)
    optimizer, scheduler = make_optimizer_and_schedule(
        kd_student.parameters(), config["lr_student"], epochs
    )

    best_bike_f1 = 0.0

    for epoch in range(epochs):
        kd_student.train()
        print(f"\n===== KD Student (ViT-Tiny) — Epoch {epoch+1}/{epochs} =====")

        for images, labels in tqdm(train_loader):
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad(set_to_none=True)

            with torch.no_grad():
                teacher_logits = teacher(images)

            student_logits = kd_student(images)

            kd_loss = kd_kl_loss(student_logits, teacher_logits, temperature)
            ce_loss = ce_loss_fn(student_logits, labels)

            loss = alpha * kd_loss + (1.0 - alpha) * ce_loss

            loss.backward()
            torch.nn.utils.clip_grad_norm_(kd_student.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

        _, _, _, f1 = evaluate(kd_student, val_loader)

        print("\n=== Validation Metrics (KD Student: ViT-Tiny) ===")
        print(f"Class 1 (bike)     — F1: {f1[1]:.4f}")
        print(f"Class 0 (non-bike) — F1: {f1[0]:.4f}")
        print(f"--> Bike F1 (selection metric): {f1[1]:.4f}")

        if f1[1] > best_bike_f1:
            best_bike_f1 = f1[1]
            torch.save(kd_student.state_dict(), save_path)
            print("Saved NEW BEST KD ViT-Tiny student!")

    print("\n===== KD Training Complete =====")
    print(f"Best Bike F1 (KD Student: ViT-Tiny): {best_bike_f1:.4f}")
    print(f"Best KD student saved at: {save_path}")

    return best_bike_f1


kd_student_path = "student_kd_vit_tiny.pth"
train_logit_kd(teacher_model, kd_student_path, config["epochs_student"])

Frozen ViT-Large teacher loaded.
KD student = ViT-Tiny loaded.

===== KD Student (ViT-Tiny) — Epoch 1/18 =====


100%|██████████| 1504/1504 [06:33<00:00,  3.82it/s]



=== Validation Metrics (KD Student: ViT-Tiny) ===
Class 1 (bike)     — F1: 0.7007
Class 0 (non-bike) — F1: 0.9958
--> Bike F1 (selection metric): 0.7007
Saved NEW BEST KD ViT-Tiny student!

===== KD Student (ViT-Tiny) — Epoch 2/18 =====


100%|██████████| 1504/1504 [06:33<00:00,  3.82it/s]



=== Validation Metrics (KD Student: ViT-Tiny) ===
Class 1 (bike)     — F1: 0.6990
Class 0 (non-bike) — F1: 0.9969
--> Bike F1 (selection metric): 0.6990

===== KD Student (ViT-Tiny) — Epoch 3/18 =====


100%|██████████| 1504/1504 [06:33<00:00,  3.82it/s]


... (27 outputs left out)

100%|██████████| 1504/1504 [06:33<00:00,  3.82it/s]



=== Validation Metrics (KD Student: ViT-Tiny) ===
Class 1 (bike)     — F1: 0.8491
Class 0 (non-bike) — F1: 0.9984
--> Bike F1 (selection metric): 0.8491

===== KD Student (ViT-Tiny) — Epoch 18/18 =====


100%|██████████| 1504/1504 [06:33<00:00,  3.82it/s]



=== Validation Metrics (KD Student: ViT-Tiny) ===
Class 1 (bike)     — F1: 0.8627
Class 0 (non-bike) — F1: 0.9986
--> Bike F1 (selection metric): 0.8627
Saved NEW BEST KD ViT-Tiny student!

===== KD Training Complete =====
Best Bike F1 (KD Student: ViT-Tiny): 0.8627
Best KD student saved at: student_kd_vit_tiny.pth


In [10]:
# =========================================================
# Test the logit-KD student (ViT-Large teacher)
# =========================================================

kd_student = load_frozen_model("vit_tiny_patch16_224", "student_kd_vit_tiny.pth")
print("Loaded KD ViT-Tiny student.")

acc, precision, recall, f1 = evaluate(kd_student, test_loader)

print("===== FINAL TEST METRICS (KD Student: ViT-Tiny) =====")
print(f"Overall Accuracy        : {acc:.4f}")
print(f"Class 1 (bike)     — Precision: {precision[1]:.4f}, Recall: {recall[1]:.4f}, F1: {f1[1]:.4f}")
print(f"Class 0 (non-bike) — Precision: {precision[0]:.4f}, Recall: {recall[0]:.4f}, F1: {f1[0]:.4f}")
print(f"--> Final Bike-Class F1 (KD Student): {f1[1]:.4f}")

Loaded KD ViT-Tiny student.
===== FINAL TEST METRICS (KD Student: ViT-Tiny) =====
Overall Accuracy        : 0.9983
Class 1 (bike)     — Precision: 0.9663, Recall: 0.8600, F1: 0.9101
Class 0 (non-bike) — Precision: 0.9986, Recall: 0.9997, F1: 0.9991
--> Final Bike-Class F1 (KD Student): 0.9101


## 1.4 Feature KD on the CLS token

Instead of matching what the teacher concludes, match what it represents. I
take the CLS token at block 3 and at the last block, project the teacher's
1024-wide token down to the student's 192 with a learnable linear layer, and
penalise the L2 distance. Logit KD and cross-entropy stay in the objective,
so this adds a signal rather than replacing one.

The projection layer is trained together with the student, and it is thrown
away afterwards: it exists only to make the two spaces comparable during
training.

In [11]:
# =========================================================
# Experiment I, step 4: feature-based KD on the CLS token
# =========================================================

teacher = load_frozen_model("vit_large_patch16_224", "vit_large_teacher_best.pth")
print("Teacher ready.")


def train_feature_kd(teacher, teacher_dim, save_path, epochs,
                     temperature=4.0, alpha=0.5, beta=1.0, gamma=0.5):
    """Logit KD plus an L2 match on the CLS token of two blocks.

    The teacher CLS token is projected down to the student width by a
    learnable linear layer, which trains together with the student. Matching
    is done at block 3 and at the last block, so the student gets both an
    early and a late view of the teacher's representation.
    """
    student = new_student()

    proj = nn.Linear(teacher_dim, 192).to(device)

    # Forward hooks are the least invasive way to read intermediate tokens
    # out of a timm model without rewriting its forward pass.
    teacher_feats = {}
    student_feats = {}

    def save_cls(store, name):
        def hook(module, input, output):
            store[name] = output[:, 0]  # token 0 is the CLS token
        return hook

    teacher.blocks[3].register_forward_hook(save_cls(teacher_feats, "block3"))
    teacher.blocks[-1].register_forward_hook(save_cls(teacher_feats, "last"))

    student.blocks[3].register_forward_hook(save_cls(student_feats, "block3"))
    student.blocks[-1].register_forward_hook(save_cls(student_feats, "last"))

    print("Hooks registered.")

    ce_loss_fn = nn.CrossEntropyLoss(weight=STUDENT_CLASS_WEIGHTS)
    mse = nn.MSELoss()

    # The projection layer is part of the optimisation, not a fixed map.
    optimizer, scheduler = make_optimizer_and_schedule(
        list(student.parameters()) + list(proj.parameters()),
        config["lr_student"],
        epochs
    )

    best_f1 = 0.0

    for epoch in range(epochs):
        student.train()
        print(f"\n===== FEATURE-KD Epoch {epoch+1}/{epochs} =====")

        for images, labels in tqdm(train_loader):
            images, labels = images.to(device), labels.to(device)

            teacher_feats.clear()
            student_feats.clear()

            optimizer.zero_grad(set_to_none=True)

            with torch.no_grad():
                teacher_logits = teacher(images)   # hooks fill teacher_feats

            student_logits = student(images)       # hooks fill student_feats

            kd_loss = kd_kl_loss(student_logits, teacher_logits, temperature)

            f_loss = 0.0
            for layer in ["block3", "last"]:
                t_feat = teacher_feats[layer]
                s_feat = student_feats[layer]

                t_proj = proj(t_feat)
                f_loss += mse(s_feat, t_proj)

            ce_loss = ce_loss_fn(student_logits, labels)

            loss = alpha * kd_loss + beta * f_loss + gamma * ce_loss

            loss.backward()
            torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

        _, _, _, f1 = evaluate(student, val_loader)

        print(f"Bike F1: {f1[1]:.4f}")

        if f1[1] > best_f1:
            best_f1 = f1[1]
            torch.save(student.state_dict(), save_path)
            print("Saved BEST Feature-KD student!")

    print("\n===== FEATURE KD Training Complete =====")
    print(f"Best Bike F1: {best_f1:.4f}")

    return best_f1


# ViT-Large has a hidden width of 1024, ViT-Tiny has 192.
train_feature_kd(
    teacher,
    teacher_dim=1024,
    save_path="student_feature_kd_vit_tiny.pth",
    epochs=config["epochs_student"],
)

Teacher ready.
Hooks registered.

===== FEATURE-KD Epoch 1/18 =====


100%|██████████| 1504/1504 [06:34<00:00,  3.82it/s]


Bike F1: 0.7258
Saved BEST Feature-KD student!

===== FEATURE-KD Epoch 2/18 =====


100%|██████████| 1504/1504 [06:34<00:00,  3.82it/s]


Bike F1: 0.5568

===== FEATURE-KD Epoch 3/18 =====


100%|██████████| 1504/1504 [06:34<00:00,  3.82it/s]


... (27 outputs left out)

100%|██████████| 1504/1504 [06:34<00:00,  3.82it/s]


Bike F1: 0.8868
Saved BEST Feature-KD student!

===== FEATURE-KD Epoch 18/18 =====


100%|██████████| 1504/1504 [06:34<00:00,  3.82it/s]


Bike F1: 0.8972
Saved BEST Feature-KD student!

===== FEATURE KD Training Complete =====
Best Bike F1: 0.8972


In [12]:
# =========================================================
# Test the feature-KD student (ViT-Large teacher)
# =========================================================

checkpoint_path = "student_feature_kd_vit_tiny.pth"
feature_kd_student = load_frozen_model("vit_tiny_patch16_224", checkpoint_path)

print(f"\nLoaded Feature-KD ViT-Tiny student model from: {checkpoint_path}\n")

acc, precision, recall, f1 = evaluate(feature_kd_student, test_loader)

print("===== FINAL TEST METRICS (Feature-KD Student: ViT-Tiny) =====")
print(f"Overall Accuracy        : {acc:.4f}")
print(f"Class 1 (bike)     — Precision: {precision[1]:.4f}, Recall: {recall[1]:.4f}, F1: {f1[1]:.4f}")
print(f"Class 0 (non-bike) — Precision: {precision[0]:.4f}, Recall: {recall[0]:.4f}, F1: {f1[0]:.4f}")
print(f"--> Final Bike-Class F1 (Feature-KD Student): {f1[1]:.4f}")


Loaded Feature-KD ViT-Tiny student model from: student_feature_kd_vit_tiny.pth

===== FINAL TEST METRICS (Feature-KD Student: ViT-Tiny) =====
Overall Accuracy        : 0.9982
Class 1 (bike)     — Precision: 0.9556, Recall: 0.8600, F1: 0.9053
Class 0 (non-bike) — Precision: 0.9986, Recall: 0.9996, F1: 0.9991
--> Final Bike-Class F1 (Feature-KD Student): 0.9053


## 1.5 Attention-map KD

Here the student is asked to look where the teacher looks. I take the
CLS-to-patch attention row from the last block, normalise both rows, and
penalise the squared difference.

Two practical problems had to be solved first. timm uses fused attention, so
the attention matrix is never returned by the module and cannot be read off a
hook directly; the hook below therefore recomputes `q @ k.T` from the block's
input tokens. And the teacher has 16 heads against the student's 3, so there
is no head-to-head pairing: averaging over heads gives both models a map of
the same shape, which is the usual way around this in the ViT distillation
literature.

In [13]:
# =========================================================
# Experiment I, step 5: attention-map KD (CLS -> patches)
# =========================================================

teacher = load_frozen_model("vit_large_patch16_224", "vit_large_teacher_best.pth")
print("Frozen ViT-Large teacher loaded.")


def extract_cls_attention(module, input, store):
    """Recompute the attention of one timm Attention block and keep the CLS row.

    timm runs fused attention, so the attention matrix is never exposed as a
    module output and cannot simply be read from a hook. Here the hook takes
    the block's input tokens and redoes q @ k.T itself.

    Averaging over heads is what makes this comparable at all: the teacher
    and the student have different head counts (16 against 3), so there is no
    head-to-head correspondence, but the head-averaged map has the same shape
    for both.
    """
    x = input[0]                      # (B, N, C)
    B, N, C = x.shape

    qkv = module.qkv(x)               # (B, N, 3*C)
    qkv = qkv.reshape(B, N, 3, module.num_heads, C // module.num_heads)
    qkv = qkv.permute(2, 0, 3, 1, 4)  # (3, B, heads, N, head_dim)
    q, k = qkv[0], qkv[1]             # v is not needed, only the attention map

    attn = (q @ k.transpose(-2, -1)) * module.scale
    attn = attn.softmax(dim=-1)       # (B, heads, N, N)

    attn = attn.mean(dim=1)           # average over heads -> (B, N, N)
    store["cls"] = attn[:, 0, :]      # the CLS row: how CLS reads the patches


def train_attention_kd(teacher, save_path, epochs,
                       temperature=4.0, alpha=0.5, delta=1.0, gamma=0.5):
    """Logit KD plus an L2 match on the last block's CLS-to-patch attention."""
    student = new_student()
    print("ViT-Tiny student initialized.")

    teacher_attn = {}
    student_attn = {}

    def teacher_hook(module, input, output):
        extract_cls_attention(module, input, teacher_attn)

    def student_hook(module, input, output):
        extract_cls_attention(module, input, student_attn)

    teacher.blocks[-1].attn.register_forward_hook(teacher_hook)
    student.blocks[-1].attn.register_forward_hook(student_hook)

    print("Attention hooks registered correctly (recomputed attention).")

    ce_loss_fn = nn.CrossEntropyLoss(weight=STUDENT_CLASS_WEIGHTS)
    optimizer, scheduler = make_optimizer_and_schedule(
        student.parameters(), config["lr_student"], epochs
    )

    best_f1 = 0.0

    for epoch in range(epochs):
        student.train()
        print(f"\n===== ATTENTION-KD Epoch {epoch+1}/{epochs} =====")

        for images, labels in tqdm(train_loader):
            images, labels = images.to(device), labels.to(device)

            teacher_attn.clear()
            student_attn.clear()

            optimizer.zero_grad(set_to_none=True)

            with torch.no_grad():
                teacher_logits = teacher(images)

            student_logits = student(images)

            logit_kd = kd_kl_loss(student_logits, teacher_logits, temperature)

            # Renormalise both rows before comparing them. Without this the
            # loss partly measures the difference in total attention mass
            # rather than where the attention goes.
            t_attn = teacher_attn["cls"]
            s_attn = student_attn["cls"]

            t_attn = t_attn / t_attn.sum(dim=-1, keepdim=True)
            s_attn = s_attn / s_attn.sum(dim=-1, keepdim=True)

            attn_kd = F.mse_loss(s_attn, t_attn)

            ce_loss = ce_loss_fn(student_logits, labels)

            loss = alpha * logit_kd + delta * attn_kd + gamma * ce_loss

            loss.backward()
            torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

        _, _, _, f1 = evaluate(student, val_loader)

        print(f"Bike F1: {f1[1]:.4f}")

        if f1[1] > best_f1:
            best_f1 = f1[1]
            torch.save(student.state_dict(), save_path)
            print("Saved BEST Attention-KD student!")

    print("\n===== ATTENTION KD Training Complete =====")
    print(f"Best Bike F1 (Attention KD): {best_f1:.4f}")
    print(f"Model saved at: {save_path}")

    return best_f1


save_path = "student_attention_kd_vit_tiny.pth"
train_attention_kd(teacher, save_path, config["epochs_student"])

Frozen ViT-Large teacher loaded.
ViT-Tiny student initialized.
Attention hooks registered correctly (recomputed attention).

===== ATTENTION-KD Epoch 1/18 =====


100%|██████████| 1504/1504 [06:39<00:00,  3.76it/s]


Bike F1: 0.8400
Saved BEST Attention-KD student!

===== ATTENTION-KD Epoch 2/18 =====


100%|██████████| 1504/1504 [06:39<00:00,  3.76it/s]


Bike F1: 0.7921

===== ATTENTION-KD Epoch 3/18 =====


100%|██████████| 1504/1504 [06:39<00:00,  3.76it/s]


... (27 outputs left out)

100%|██████████| 1504/1504 [06:39<00:00,  3.76it/s]


Bike F1: 0.8952
Saved BEST Attention-KD student!

===== ATTENTION-KD Epoch 18/18 =====


100%|██████████| 1504/1504 [06:39<00:00,  3.76it/s]


Bike F1: 0.8571

===== ATTENTION KD Training Complete =====
Best Bike F1 (Attention KD): 0.8952
Model saved at: student_attention_kd_vit_tiny.pth


In [14]:
# =========================================================
# Test the attention-KD student (ViT-Large teacher)
# =========================================================

checkpoint_path = "student_attention_kd_vit_tiny.pth"
attention_kd_student = load_frozen_model("vit_tiny_patch16_224", checkpoint_path)

print(f"\nLoaded Attention-KD ViT-Tiny student model from: {checkpoint_path}\n")

acc, precision, recall, f1 = evaluate(attention_kd_student, test_loader)

print("===== FINAL TEST METRICS (Attention-KD Student: ViT-Tiny) =====")
print(f"Overall Accuracy        : {acc:.4f}")
print(f"Class 1 (bike)     — Precision: {precision[1]:.4f}, Recall: {recall[1]:.4f}, F1: {f1[1]:.4f}")
print(f"Class 0 (non-bike) — Precision: {precision[0]:.4f}, Recall: {recall[0]:.4f}, F1: {f1[0]:.4f}")
print(f"--> Final Bike-Class F1 (Attention-KD Student): {f1[1]:.4f}")


Loaded Attention-KD ViT-Tiny student model from: student_attention_kd_vit_tiny.pth

===== FINAL TEST METRICS (Attention-KD Student: ViT-Tiny) =====
Overall Accuracy        : 0.9981
Class 1 (bike)     — Precision: 0.9451, Recall: 0.8600, F1: 0.9005
Class 0 (non-bike) — Precision: 0.9986, Recall: 0.9995, F1: 0.9990
--> Final Bike-Class F1 (Attention-KD Student): 0.9005


### Experiment I, test-set results

| Model | Precision | Recall | F1 |
| --- | --- | --- | --- |
| Teacher (ViT-Large) | 1.000 | 0.960 | 0.980 |
| Student, no KD | 0.920 | 0.800 | 0.856 |
| Logit KD | 0.966 | 0.860 | 0.910 |
| Feature KD | 0.956 | 0.860 | 0.905 |
| Attention KD | 0.945 | 0.860 | 0.901 |

All three signals beat the baseline, and almost all of the gain is in
precision: recall moves from 0.80 to 0.86 and then stops at 0.86 for all
three. The simplest signal, plain logits, comes out on top. My reading is
that the student is too narrow to absorb the teacher's 1024-wide features or
its 16-head attention pattern, so those richer signals mostly add noise,
while a sharpened decision boundary is something a 5.5M-parameter model can
actually use.

---

# Part 2 - Experiment II: does a smaller teacher change the answer?

Same three distillation runs, with ViT-Base (85.8M) as the teacher instead of
ViT-Large (303.3M). If the capacity gap is what holds the student back, a
closer teacher should help, and feature matching in particular should get
easier: the projection now bridges 768 to 192 rather than 1024 to 192.

The teacher is fine-tuned by the same `finetune_teacher` function used in
Part 1, so the only deliberate difference between the two parts is the
teacher itself.

In [15]:
# =========================================================
# Experiment II, step 1a: load the ViT-Base teacher
# =========================================================

# Same recipe as Experiment I, with ViT-Base in place of ViT-Large. Nothing
# else about the setup changes, so any difference downstream is down to the
# teacher's capacity.
teacher_model_base = timm.create_model(
    'vit_base_patch16_224',
    pretrained=True
)

teacher_model_base.head = nn.Linear(
    teacher_model_base.head.in_features,
    config["num_classes"]
)

teacher_model_base = teacher_model_base.to(device)

teacher_model_base.eval()

for p in teacher_model_base.parameters():
    p.requires_grad = False

param_count, flops_str = model_cost(teacher_model_base)

print("ViT-Base teacher model initialized.")
print(f"Parameters: {param_count/1e6:.2f}M")
print(f"FLOPs: {flops_str}")
print("Classification head: Binary")
print(f"Device: {device}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
blocks.0.attn.attn_drop, blocks.1.attn.attn_drop, blocks.10.attn.attn_drop, blocks.11.attn.attn_drop, blocks.2.attn.attn_drop, blocks.3.attn.attn_drop, blocks.4.attn.attn_drop, blocks.5.attn.attn_drop, blocks.6.attn.attn_drop, blocks.7.attn.attn_drop, blocks.8.attn.attn_drop, blocks.9.attn.attn_drop


ViT-Base teacher model initialized.
Parameters: 85.80M
FLOPs: 16.87 GFLOPs
Classification head: Binary
Device: cuda


In [16]:
# =========================================================
# Experiment II, step 1b: fine-tune the ViT-Base teacher
# =========================================================

teacher_base_save_path = "vit_base_teacher_best.pth"
finetune_teacher(
    teacher_model_base,
    label="ViT-Base",
    save_path=teacher_base_save_path,
    epochs=config["epochs_teacher"],
)


===== Epoch 1/4 =====


100%|██████████| 1407/1407 [05:13<00:00,  4.48it/s]

Train Loss: 0.1512 | Train Acc: 0.9962



=== Validation Metrics ===
Class 1 (bike)     — Precision: 0.8852, Recall: 0.9474, F1: 0.9153
Class 0 (non-bike) — Precision: 0.9994, Recall: 0.9986, F1: 0.9990
 Saved NEW BEST ViT-Base teacher model

===== Epoch 2/4 =====


100%|██████████| 1407/1407 [05:13<00:00,  4.49it/s]

Train Loss: 0.0562 | Train Acc: 0.9984


... (3 outputs left out)


=== Validation Metrics ===
Class 1 (bike)     — Precision: 0.9815, Recall: 0.9298, F1: 0.9550
Class 0 (non-bike) — Precision: 0.9992, Recall: 0.9998, F1: 0.9995
 Saved NEW BEST ViT-Base teacher model

===== Epoch 4/4 =====


100%|██████████| 1407/1407 [05:13<00:00,  4.49it/s]

Train Loss: 0.0030 | Train Acc: 0.9999



=== Validation Metrics ===
Class 1 (bike)     — Precision: 0.9649, Recall: 0.9649, F1: 0.9649
Class 0 (non-bike) — Precision: 0.9996, Recall: 0.9996, F1: 0.9996
 Saved NEW BEST ViT-Base teacher model

===== Teacher Fine-Tuning Complete =====
Best Bike F1 achieved: 0.9649
Best model saved at: vit_base_teacher_best.pth


In [17]:
# =========================================================
# Experiment II, step 1c: test the ViT-Base teacher
# =========================================================

best_model_path = "vit_base_teacher_best.pth"

try:
    state_dict = torch.load(best_model_path, map_location=device)
    teacher_model_base.load_state_dict(state_dict)
    print(f"\nLoaded BEST ViT-Base teacher model from: {best_model_path}")
except Exception as e:
    print(f"ERROR loading checkpoint: {e}")
    raise

for p in teacher_model_base.parameters():
    p.requires_grad = False

teacher_model_base.eval()

acc, precision, recall, f1 = evaluate(teacher_model_base, test_loader)

print("\n===== FINAL TEST METRICS (ViT-Best TEACHER) =====")
print(f"Overall Accuracy            : {acc:.4f}")
print(f"Class 1 (bike)     — Precision: {precision[1]:.4f}, Recall: {recall[1]:.4f}, F1: {f1[1]:.4f}")
print(f"Class 0 (non-bike) — Precision: {precision[0]:.4f}, Recall: {recall[0]:.4f}, F1: {f1[0]:.4f}")
print(f"--> Final Bike-Class F1 (Teacher): {f1[1]:.4f}")


Loaded BEST ViT-Base teacher model from: vit_base_teacher_best.pth

===== FINAL TEST METRICS (ViT-Best TEACHER) =====
Overall Accuracy            : 0.9988
Class 1 (bike)     — Precision: 0.9314, Recall: 0.9500, F1: 0.9406
Class 0 (non-bike) — Precision: 0.9995, Recall: 0.9993, F1: 0.9994
--> Final Bike-Class F1 (Teacher): 0.9406


## 2.2 Baseline student, retrained

The baseline does not involve the teacher, so Part 1's number would have
done. I retrained it in this session anyway, and it is useful that I did: the
two runs do not land in the same place, which says something about how much
of a small F1 difference is real. Both numbers are reported in the write-up.

In [18]:
# =========================================================
# Experiment II, step 2a: baseline student, trained again
# =========================================================

# The baseline does not depend on the teacher, so in principle the number
# from Experiment I could be reused. I retrained it in this session anyway,
# which turned out to be worth doing: the two runs land at a different
# bike-class F1 even under the same seed, and that spread is a useful piece
# of context for reading the KD gains. Both runs are reported.
train_baseline_student("student_vit_tiny_baseline.pth", config["epochs_student"])

blocks.0.attn.attn_drop, blocks.1.attn.attn_drop, blocks.10.attn.attn_drop, blocks.11.attn.attn_drop, blocks.2.attn.attn_drop, blocks.3.attn.attn_drop, blocks.4.attn.attn_drop, blocks.5.attn.attn_drop, blocks.6.attn.attn_drop, blocks.7.attn.attn_drop, blocks.8.attn.attn_drop, blocks.9.attn.attn_drop


Loaded ViT-Tiny student with 5.52M params.
Student inference FLOPs: 1.08 GFLOPs

===== Student Baseline (ViT-Tiny) — Epoch 1/18 =====


100%|██████████| 1504/1504 [01:06<00:00, 22.65it/s]

Train Loss: 0.1469 | Train Acc: 0.9764


=== Validation Metrics ===
Accuracy                 : 0.9932
Class 1 (bike)     — Precision: 0.7170, Recall: 0.6667, F1: 0.6909
Class 0 (non-bike) — Precision: 0.9962, Recall: 0.9970, F1: 0.9966
 Saved NEW BEST ViT-Tiny baseline model

===== Student Baseline (ViT-Tiny) — Epoch 2/18 =====


100%|██████████| 1504/1504 [01:06<00:00, 22.76it/s]

... (46 outputs left out)

=== Validation Metrics ===
Accuracy                 : 0.9968
Class 1 (bike)     — Precision: 0.9362, Recall: 0.7719, F1: 0.8462
Class 0 (non-bike) — Precision: 0.9974, Recall: 0.9994, F1: 0.9984
 Saved NEW BEST ViT-Tiny baseline model

===== Student Baseline (ViT-Tiny) — Epoch 18/18 =====


100%|██████████| 1504/1504 [01:05<00:00, 22.88it/s]

Train Loss: 0.0022 | Train Acc: 0.9997


=== Validation Metrics ===
Accuracy                 : 0.9968
Class 1 (bike)     — Precision: 0.9362, Recall: 0.7719, F1: 0.8462
Class 0 (non-bike) — Precision: 0.9974, Recall: 0.9994, F1: 0.9984

===== Student Baseline Training Complete =====
Best Bike F1 (Student Baseline): 0.8462
Best model saved at: student_vit_tiny_baseline.pth


In [19]:
# =========================================================
# Experiment II, step 2b: test the retrained baseline
# =========================================================

checkpoint_path = "student_vit_tiny_baseline.pth"
student_test_model = load_frozen_model("vit_tiny_patch16_224", checkpoint_path)

print(f"Loaded best ViT-Tiny student model from: {checkpoint_path}")

acc, precision, recall, f1 = evaluate(student_test_model, test_loader)

print("\n===== FINAL TEST METRICS (STUDENT: ViT-Tiny) =====")
print(f"Overall Accuracy        : {acc:.4f}")
print(f"Class 1 (bike)     — Precision: {precision[1]:.4f}, Recall: {recall[1]:.4f}, F1: {f1[1]:.4f}")
print(f"Class 0 (non-bike) — Precision: {precision[0]:.4f}, Recall: {recall[0]:.4f}, F1: {f1[0]:.4f}")
print(f"--> Final Bike-Class F1 (Student): {f1[1]:.4f}")

Loaded best ViT-Tiny student model from: student_vit_tiny_baseline.pth

===== FINAL TEST METRICS (STUDENT: ViT-Tiny) =====
Overall Accuracy        : 0.9978
Class 1 (bike)     — Precision: 0.9432, Recall: 0.8300, F1: 0.8830
Class 0 (non-bike) — Precision: 0.9983, Recall: 0.9995, F1: 0.9989
--> Final Bike-Class F1 (Student): 0.8830


## 2.3 The three KD runs with the ViT-Base teacher

These cells call the same three functions defined in Part 1. Only the teacher
and, for feature KD, the projection width change.

In [20]:
# =========================================================
# Experiment II, step 3: logit-based KD (ViT-Base -> ViT-Tiny)
# =========================================================

teacher_model = load_frozen_model("vit_base_patch16_224", "vit_base_teacher_best.pth")
print("Frozen ViT-Base teacher loaded.")

kd_student_path = "student_kd_vit_tiny.pth"
train_logit_kd(teacher_model, kd_student_path, config["epochs_student"])

Frozen ViT-Base teacher loaded.
KD student = ViT-Tiny loaded.

===== KD Student (ViT-Tiny) — Epoch 1/18 =====


100%|██████████| 1504/1504 [02:26<00:00, 10.28it/s]



=== Validation Metrics (KD Student: ViT-Tiny) ===
Class 1 (bike)     — F1: 0.7037
Class 0 (non-bike) — F1: 0.9968
--> Bike F1 (selection metric): 0.7037
Saved NEW BEST KD ViT-Tiny student!

===== KD Student (ViT-Tiny) — Epoch 2/18 =====


100%|██████████| 1504/1504 [02:26<00:00, 10.28it/s]



=== Validation Metrics (KD Student: ViT-Tiny) ===
Class 1 (bike)     — F1: 0.6463
Class 0 (non-bike) — F1: 0.9941
--> Bike F1 (selection metric): 0.6463

===== KD Student (ViT-Tiny) — Epoch 3/18 =====


100%|██████████| 1504/1504 [02:26<00:00, 10.28it/s]


... (27 outputs left out)

100%|██████████| 1504/1504 [02:26<00:00, 10.28it/s]



=== Validation Metrics (KD Student: ViT-Tiny) ===
Class 1 (bike)     — F1: 0.8727
Class 0 (non-bike) — F1: 0.9986
--> Bike F1 (selection metric): 0.8727

===== KD Student (ViT-Tiny) — Epoch 18/18 =====


100%|██████████| 1504/1504 [02:26<00:00, 10.28it/s]



=== Validation Metrics (KD Student: ViT-Tiny) ===
Class 1 (bike)     — F1: 0.8889
Class 0 (non-bike) — F1: 0.9988
--> Bike F1 (selection metric): 0.8889
Saved NEW BEST KD ViT-Tiny student!

===== KD Training Complete =====
Best Bike F1 (KD Student: ViT-Tiny): 0.8889
Best KD student saved at: student_kd_vit_tiny.pth


In [21]:
# =========================================================
# Test the logit-KD student (ViT-Base teacher)
# =========================================================

kd_student = load_frozen_model("vit_tiny_patch16_224", "student_kd_vit_tiny.pth")
print("Loaded KD ViT-Tiny student.")

acc, precision, recall, f1 = evaluate(kd_student, test_loader)

print("===== FINAL TEST METRICS (KD Student: ViT-Tiny) =====")
print(f"Overall Accuracy        : {acc:.4f}")
print(f"Class 1 (bike)     — Precision: {precision[1]:.4f}, Recall: {recall[1]:.4f}, F1: {f1[1]:.4f}")
print(f"Class 0 (non-bike) — Precision: {precision[0]:.4f}, Recall: {recall[0]:.4f}, F1: {f1[0]:.4f}")
print(f"--> Final Bike-Class F1 (KD Student): {f1[1]:.4f}")

Loaded KD ViT-Tiny student.
===== FINAL TEST METRICS (KD Student: ViT-Tiny) =====
Overall Accuracy        : 0.9985
Class 1 (bike)     — Precision: 0.9670, Recall: 0.8800, F1: 0.9215
Class 0 (non-bike) — Precision: 0.9988, Recall: 0.9997, F1: 0.9992
--> Final Bike-Class F1 (KD Student): 0.9215


In [22]:
# =========================================================
# Experiment II, step 4: feature-based KD (ViT-Base -> ViT-Tiny)
# =========================================================

teacher = load_frozen_model("vit_base_patch16_224", "vit_base_teacher_best.pth")
print("Teacher ready.")

# ViT-Base is 768-wide, so the projection is a smaller jump than the 1024
# of Experiment I.
train_feature_kd(
    teacher,
    teacher_dim=768,
    save_path="student_feature_kd_vit_tiny.pth",
    epochs=config["epochs_student"],
)

Teacher ready.
Hooks registered.

===== FEATURE-KD Epoch 1/18 =====


100%|██████████| 1504/1504 [02:26<00:00, 10.25it/s]


Bike F1: 0.5896
Saved BEST Feature-KD student!

===== FEATURE-KD Epoch 2/18 =====


100%|██████████| 1504/1504 [02:26<00:00, 10.26it/s]


Bike F1: 0.7368
Saved BEST Feature-KD student!

===== FEATURE-KD Epoch 3/18 =====


100%|██████████| 1504/1504 [02:26<00:00, 10.26it/s]


... (27 outputs left out)

100%|██████████| 1504/1504 [02:26<00:00, 10.26it/s]


Bike F1: 0.8785

===== FEATURE-KD Epoch 18/18 =====


100%|██████████| 1504/1504 [02:26<00:00, 10.26it/s]


Bike F1: 0.8762

===== FEATURE KD Training Complete =====
Best Bike F1: 0.8868


In [23]:
# =========================================================
# Test the feature-KD student (ViT-Base teacher)
# =========================================================

checkpoint_path = "student_feature_kd_vit_tiny.pth"
feature_kd_student = load_frozen_model("vit_tiny_patch16_224", checkpoint_path)

print(f"\nLoaded Feature-KD ViT-Tiny student model from: {checkpoint_path}\n")

acc, precision, recall, f1 = evaluate(feature_kd_student, test_loader)

print("===== FINAL TEST METRICS (Feature-KD Student: ViT-Tiny) =====")
print(f"Overall Accuracy        : {acc:.4f}")
print(f"Class 1 (bike)     — Precision: {precision[1]:.4f}, Recall: {recall[1]:.4f}, F1: {f1[1]:.4f}")
print(f"Class 0 (non-bike) — Precision: {precision[0]:.4f}, Recall: {recall[0]:.4f}, F1: {f1[0]:.4f}")
print(f"--> Final Bike-Class F1 (Feature-KD Student): {f1[1]:.4f}")


Loaded Feature-KD ViT-Tiny student model from: student_feature_kd_vit_tiny.pth

===== FINAL TEST METRICS (Feature-KD Student: ViT-Tiny) =====
Overall Accuracy        : 0.9980
Class 1 (bike)     — Precision: 0.9167, Recall: 0.8800, F1: 0.8980
Class 0 (non-bike) — Precision: 0.9988, Recall: 0.9992, F1: 0.9990
--> Final Bike-Class F1 (Feature-KD Student): 0.8980


In [24]:
# =========================================================
# Experiment II, step 5: attention-map KD (ViT-Base -> ViT-Tiny)
# =========================================================

teacher = load_frozen_model("vit_base_patch16_224", "vit_base_teacher_best.pth")
print("Frozen ViT-Base teacher loaded.")

save_path = "student_attention_kd_vit_tiny.pth"
train_attention_kd(teacher, save_path, config["epochs_student"])

Frozen ViT-Base teacher loaded.
ViT-Tiny student initialized.
Attention hooks registered correctly (recomputed attention).

===== ATTENTION-KD Epoch 1/18 =====


100%|██████████| 1504/1504 [02:30<00:00, 10.00it/s]


Bike F1: 0.7778
Saved BEST Attention-KD student!

===== ATTENTION-KD Epoch 2/18 =====


100%|██████████| 1504/1504 [02:30<00:00, 10.00it/s]


Bike F1: 0.6225

===== ATTENTION-KD Epoch 3/18 =====


100%|██████████| 1504/1504 [02:30<00:00, 10.00it/s]


... (27 outputs left out)

100%|██████████| 1504/1504 [02:30<00:00, 10.00it/s]


Bike F1: 0.8598

===== ATTENTION-KD Epoch 18/18 =====


100%|██████████| 1504/1504 [02:30<00:00, 10.00it/s]


Bike F1: 0.8762

===== ATTENTION KD Training Complete =====
Best Bike F1 (Attention KD): 0.8991
Model saved at: student_attention_kd_vit_tiny.pth


In [25]:
# =========================================================
# Test the attention-KD student (ViT-Base teacher)
# =========================================================

checkpoint_path = "student_attention_kd_vit_tiny.pth"
attention_kd_student = load_frozen_model("vit_tiny_patch16_224", checkpoint_path)

print(f"\nLoaded Attention-KD ViT-Tiny student model from: {checkpoint_path}\n")

acc, precision, recall, f1 = evaluate(attention_kd_student, test_loader)

print("===== FINAL TEST METRICS (Attention-KD Student: ViT-Tiny) =====")
print(f"Overall Accuracy        : {acc:.4f}")
print(f"Class 1 (bike)     — Precision: {precision[1]:.4f}, Recall: {recall[1]:.4f}, F1: {f1[1]:.4f}")
print(f"Class 0 (non-bike) — Precision: {precision[0]:.4f}, Recall: {recall[0]:.4f}, F1: {f1[0]:.4f}")
print(f"--> Final Bike-Class F1 (Attention-KD Student): {f1[1]:.4f}")


Loaded Attention-KD ViT-Tiny student model from: student_attention_kd_vit_tiny.pth

===== FINAL TEST METRICS (Attention-KD Student: ViT-Tiny) =====
Overall Accuracy        : 0.9981
Class 1 (bike)     — Precision: 0.9655, Recall: 0.8400, F1: 0.8984
Class 0 (non-bike) — Precision: 0.9984, Recall: 0.9997, F1: 0.9990
--> Final Bike-Class F1 (Attention-KD Student): 0.8984


### Experiment II, test-set results

| Model | Precision | Recall | F1 |
| --- | --- | --- | --- |
| Teacher (ViT-Base) | 0.931 | 0.950 | 0.941 |
| Student, no KD (this session) | 0.943 | 0.830 | 0.883 |
| Logit KD | 0.967 | 0.880 | 0.922 |
| Feature KD | 0.917 | 0.880 | 0.898 |
| Attention KD | 0.966 | 0.840 | 0.898 |

The weaker teacher did not produce a weaker student. Logit KD again wins, and
at 0.922 it is the best student in the whole project, slightly ahead of the
0.910 it reached with ViT-Large. The ranking of the three signals is
unchanged, so the conclusion from Part 1 is not an artefact of one particular
teacher. Closing the capacity gap also did not rescue feature matching, which
is evidence that the student's width, not the size of the projection, is the
binding constraint.

---

# Part 3 - Experiment III: teacher assistant

The teacher-assistant idea (Mirzadeh et al., 2020) is to stop asking one
model to learn directly from a much larger one. Knowledge goes down in two
hops instead: ViT-Large teaches a ViT-Small assistant, then the assistant
teaches ViT-Tiny. The student never sees the ViT-Large teacher.

Both hops use logit KD, since Part 1 showed that to be the strongest signal,
with the same 0.7 / 0.3 weighting in each stage.

In [ ]:
# =========================================================
# Experiment III, stage 1: ViT-Large -> ViT-Small assistant
# =========================================================

# Stage one of the teacher-assistant chain. The assistant is trained exactly
# like a student, with logit KD from the ViT-Large teacher; it only becomes a
# teacher in the next cell.
teacher = load_frozen_model("vit_large_patch16_224", "vit_large_teacher_best.pth")
print("Frozen ViT-Large teacher loaded.")

ta_model = timm.create_model(
    "vit_small_patch16_224",
    pretrained=True,
    num_classes=config["num_classes"]
).to(device)

print("Teacher Assistant (ViT-Small) initialized.")

ce_loss_fn = nn.CrossEntropyLoss(weight=STUDENT_CLASS_WEIGHTS)

optimizer, scheduler = make_optimizer_and_schedule(
    ta_model.parameters(), config["lr_student"], config["epochs_student"]
)

# Here the two loss weights do not add up to 1, unlike the alpha / (1 - alpha)
# split used for the direct KD runs. Keeping 0.7 / 0.3 in both stages of the
# chain is what matters for the comparison.
T = 4.0
alpha = 0.7
gamma = 0.3

best_f1 = 0.0
ta_save_path = "vit_small_teacher_assistant.pth"

for epoch in range(config["epochs_student"]):
    ta_model.train()
    print(f"\n===== TA-KD Epoch {epoch+1}/{config['epochs_student']} =====")

    for images, labels in tqdm(train_loader):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)

        with torch.no_grad():
            teacher_logits = teacher(images)

        ta_logits = ta_model(images)

        kd_loss = kd_kl_loss(ta_logits, teacher_logits, T)
        ce_loss = ce_loss_fn(ta_logits, labels)

        loss = alpha * kd_loss + gamma * ce_loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(ta_model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

    _, _, _, f1 = evaluate(ta_model, val_loader)

    print(f"Bike F1 (TA): {f1[1]:.4f}")

    if f1[1] > best_f1:
        best_f1 = f1[1]
        torch.save(ta_model.state_dict(), ta_save_path)
        print("Saved BEST Teacher Assistant (ViT-Small).")

print("\n===== TA TRAINING COMPLETE =====")
print(f"Best Bike F1 (TA): {best_f1:.4f}")
print(f"Teacher Assistant saved at: {ta_save_path}")

In [27]:
# =========================================================
# Experiment III, stage 2: ViT-Small assistant -> ViT-Tiny
# =========================================================

# Stage two. The student never sees the ViT-Large teacher: its only source of
# soft labels is the assistant trained in the previous cell.
ta_teacher = load_frozen_model("vit_small_patch16_224", "vit_small_teacher_assistant.pth")
print("Frozen Teacher Assistant (ViT-Small) loaded.")

student = new_student()
print("Student (ViT-Tiny) initialized for TA-KD.")

ce_loss_fn = nn.CrossEntropyLoss(weight=STUDENT_CLASS_WEIGHTS)

optimizer, scheduler = make_optimizer_and_schedule(
    student.parameters(), config["lr_student"], config["epochs_student"]
)

T = 4.0
alpha = 0.7
gamma = 0.3

best_f1 = 0.0
student_save_path = "vit_tiny_ta_kd.pth"

for epoch in range(config["epochs_student"]):
    student.train()
    print(f"\n===== TA-KD (Tiny) Epoch {epoch+1}/{config['epochs_student']} =====")

    for images, labels in tqdm(train_loader):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)

        with torch.no_grad():
            ta_logits = ta_teacher(images)

        student_logits = student(images)

        kd_loss = kd_kl_loss(student_logits, ta_logits, T)
        ce_loss = ce_loss_fn(student_logits, labels)

        loss = alpha * kd_loss + gamma * ce_loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

    _, _, _, f1 = evaluate(student, val_loader)

    print(f"Bike F1 (TA-KD Student): {f1[1]:.4f}")

    if f1[1] > best_f1:
        best_f1 = f1[1]
        torch.save(student.state_dict(), student_save_path)
        print("Saved BEST TA-KD Student (ViT-Tiny).")

print("\n===== TA-KD STUDENT TRAINING COMPLETE =====")
print(f"Best Bike F1 (TA-KD Student): {best_f1:.4f}")
print(f"TA-KD student saved at: {student_save_path}")

Frozen Teacher Assistant (ViT-Small) loaded.
Student (ViT-Tiny) initialized for TA-KD.

===== TA-KD (Tiny) Epoch 1/18 =====


100%|██████████| 1504/1504 [01:17<00:00, 19.30it/s]


Bike F1 (TA-KD Student): 0.6194
Saved BEST TA-KD Student (ViT-Tiny).

===== TA-KD (Tiny) Epoch 2/18 =====


100%|██████████| 1504/1504 [01:17<00:00, 19.31it/s]


Bike F1 (TA-KD Student): 0.6423
Saved BEST TA-KD Student (ViT-Tiny).

===== TA-KD (Tiny) Epoch 3/18 =====


100%|██████████| 1504/1504 [01:18<00:00, 19.28it/s]


... (27 outputs left out)

100%|██████████| 1504/1504 [01:17<00:00, 19.31it/s]


Bike F1 (TA-KD Student): 0.8431

===== TA-KD (Tiny) Epoch 18/18 =====


100%|██████████| 1504/1504 [01:18<00:00, 19.28it/s]


Bike F1 (TA-KD Student): 0.8654

===== TA-KD STUDENT TRAINING COMPLETE =====
Best Bike F1 (TA-KD Student): 0.8762
TA-KD student saved at: vit_tiny_ta_kd.pth


In [28]:
# =========================================================
# Test the TA-KD student
# =========================================================

checkpoint_path = "vit_tiny_ta_kd.pth"
ta_kd_student = load_frozen_model("vit_tiny_patch16_224", checkpoint_path)

print(f"\nLoaded TA-KD ViT-Tiny student model from: {checkpoint_path}\n")

acc, precision, recall, f1 = evaluate(ta_kd_student, test_loader)

print("===== FINAL TEST METRICS (TA-KD Student: ViT-Tiny) =====")
print(f"Overall Accuracy        : {acc:.4f}")
print(f"Class 1 (bike)     — Precision: {precision[1]:.4f}, Recall: {recall[1]:.4f}, F1: {f1[1]:.4f}")
print(f"Class 0 (non-bike) — Precision: {precision[0]:.4f}, Recall: {recall[0]:.4f}, F1: {f1[0]:.4f}")
print(f"--> Final Bike-Class F1 (TA-KD Student): {f1[1]:.4f}")


Loaded TA-KD ViT-Tiny student model from: vit_tiny_ta_kd.pth

===== FINAL TEST METRICS (TA-KD Student: ViT-Tiny) =====
Overall Accuracy        : 0.9980
Class 1 (bike)     — Precision: 0.9444, Recall: 0.8500, F1: 0.8947
Class 0 (non-bike) — Precision: 0.9985, Recall: 0.9995, F1: 0.9990
--> Final Bike-Class F1 (TA-KD Student): 0.8947


---

# Results and what I take from them

All numbers are on the 10,000-image test set, for the bike class. Each
student is a ViT-Tiny: 5.52M parameters and 1.08 GFLOPs per image, against
the teacher's 303.3M and 59.7.

| Experiment | Model | Precision | Recall | F1 |
| --- | --- | --- | --- | --- |
| - | Teacher, ViT-Large | 1.000 | 0.960 | 0.980 |
| - | Teacher, ViT-Base | 0.931 | 0.950 | 0.941 |
| - | Student, no KD | 0.920 | 0.800 | 0.856 |
| I | Logit KD, from ViT-Large | 0.966 | 0.860 | 0.910 |
| I | Feature KD, from ViT-Large | 0.956 | 0.860 | 0.905 |
| I | Attention KD, from ViT-Large | 0.945 | 0.860 | 0.901 |
| II | Logit KD, from ViT-Base | 0.967 | 0.880 | 0.922 |
| II | Feature KD, from ViT-Base | 0.917 | 0.880 | 0.898 |
| II | Attention KD, from ViT-Base | 0.966 | 0.840 | 0.898 |
| III | Logit KD via ViT-Small assistant | 0.944 | 0.850 | 0.895 |

Three things came out of this:

1. Distillation helps, and it helps most where it matters. Every distilled
   student beats the 0.856 baseline, and the gain is concentrated in
   precision, which here means fewer false alarms about a cyclist who is not
   there.
2. The simplest signal wins. Logit KD beats feature and attention matching
   under both teachers. At 32x32 input and 1% positives, a ViT-Tiny seems
   unable to make use of a wide teacher's internal representations, so the
   extra terms cost more than they return.
3. Narrowing the capacity gap did not pay off, neither by shrinking the
   teacher nor by routing through an assistant. The assistant run reaches
   0.895, below the 0.910 of direct logit KD from the same teacher. For a rare
   safety-critical class, direct access to the strongest teacher's soft labels
   appears to be worth more than architectural proximity.

For anyone picking a method off this table for an embedded target: start with
logit KD. It was the cheapest to implement, had the fewest moving parts, and
gave the best student in every configuration I tried.

The limitations are in the write-up, and they are real ones: 32x32 CIFAR
images are not a driving scene, every number here is a single run rather than
a mean over seeds, and the hyperparameters were set by hand under a fixed
compute budget rather than searched.

## One note on re-running

Each experiment writes its student checkpoints under the same filenames
(`student_kd_vit_tiny.pth` and so on) and is evaluated immediately after it
is trained, which is how the results above were produced. If the notebook is
run end to end, only the last experiment's checkpoint files survive on disk.
Rename the paths per experiment if you want to keep all of them.